In [2]:
!pip install SciPy

In [4]:
import numpy as np
import pandas as pd
from random import randrange
from sklearn.base import clone, is_classifier, is_regressor
from scipy.stats import mode

In [22]:
class Bagging:  
    def fit(self, model, X, y, n_estimators):
        fitted_models = []
        for n in range(n_estimators):
            indices = np.random.choice(
                len(X),
                size=len(X),
                replace=True
                )
            X_boot = X.iloc[indices]
            y_boot = y.iloc[indices]
            model_copy = clone(model)
            model_copy.fit(X_boot, y_boot)
            fitted_models.append(model_copy)
        return fitted_models
    def predict(self, fitted_models, X):
        predictions = []
        for fitted_model in fitted_models:
            pred = fitted_model.predict(X)
            predictions.append(pred)
        predictions = np.array(predictions)
        if is_regressor(fitted_models[0]):#наивно полагаю, что все модели будут одинаковыми, я же их отклонировал
            return np.mean(predictions, axis=0)
        if is_classifier(fitted_models[0]):
            return mode(predictions, axis=0).mode
            

In [23]:
class Boosting:  
    def fit(self, model, X, y, n_estimators, learning_rate):
        self.learning_rate = learning_rate
        self.initial_prediction = np.mean(y)
        fitted_models = []
        predictions = np.full(len(y), self.initial_prediction)
        for n in range(n_estimators):
            residuals = y - predictions
            model_copy = clone(model)
            model_copy.fit(X, residuals)
            predictions += learning_rate * model_copy.predict(X)
            fitted_models.append(model_copy)
        return fitted_models
        
    def predict(self, fitted_models, X):
        predictions = np.full(len(X), self.initial_prediction)
    
        for model in fitted_models:
            predictions += self.learning_rate * model.predict(X)
    
        return predictions

Теперь возьмём какой-нибудь рофлодатасет и погоняем через рофлобеггинг и рофлобустинг рофломодели


In [24]:
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Target'] = data.target
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
 8   Target      20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


мне лень делать нормальный EDA, я этим наверное займусь немного позже. Щас просто модельки погоняем. Гиперпараметры подбирать не буду


In [25]:
X = df.drop(columns=['Target'])
y = df['Target']


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import root_mean_squared_error

Внезапно, одно решающее дерево будет что-то типа нашего бейзлайна

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = DecisionTreeRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(root_mean_squared_error(y_test, y_pred))

0.7306795578317605


In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = GradientBoostingRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(root_mean_squared_error(y_test, y_pred))

0.5378214924164503


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
bagging = Bagging()
fitted_models = bagging.fit(
    DecisionTreeRegressor(),
    X_train,
    y_train,
    n_estimators=50
)
y_pred = bagging.predict(
    fitted_models,
    X_test
)
print(root_mean_squared_error(y_test, y_pred))

0.5086239216690377


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
boosting = Boosting()
fitted_models = boosting.fit(
    DecisionTreeRegressor(),
    X_train,
    y_train,
    n_estimators=50,
    learning_rate=0.1
)
y_pred = boosting.predict(
    fitted_models,
    X_test
)
print(root_mean_squared_error(y_test, y_pred))

0.6951465490303514


(я малость проиграл градиентному бустингу из sklearn, но я же могу не дерево засунуть в градиентный бустинг, а линрег)

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(root_mean_squared_error(y_test, y_pred))

0.7356145375446766


In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
model = ElasticNet()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(root_mean_squared_error(y_test, y_pred))

0.8742232571189948


(всё становится очень странно, почему дерево обгоняет линрег, а линрег с регуляризациями отстаёт от линрега без них)

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
bagging = Bagging()
fitted_models = bagging.fit(
    LinearRegression(),
    X_train,
    y_train,
    n_estimators=50
)
y_pred = bagging.predict(
    fitted_models,
    X_test
)
print(root_mean_squared_error(y_test, y_pred))

0.7348169705621985


In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
boosting = Boosting()
fitted_models = boosting.fit(
    LinearRegression(),
    X_train,
    y_train,
    n_estimators=50,
    learning_rate=0.1
)
y_pred = boosting.predict(
    fitted_models,
    X_test
)
print(root_mean_squared_error(y_test, y_pred))

0.7354788260603831


вывод: бустить линейные модели бесполезно, они сами прекрасно со всем справляются и вообще умнички
а беггинг крутой, зря вы его недооцениваете

| Model                      | RMSE |
| -------------------------- | ---: |
| Linear Regression          |  0.74 |
| ElasticNet                 |  0.87 |
| Decision Tree              |  0.73 |
| My Bagging                 |  0.51 |
| My Boosting                |  0.70 |
| sklearn Gradient Boosting  |  0.54 |
| Bagging Linear Regression  |  0.73 |
| Boosting Linear Regression |  0.74 |
